# Loan Eligibility Prediction
**Algorithm:** Gaussian Naive Bayes

> **Bug-fixed version** — feature order and preprocessing aligned with app.py

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
%matplotlib inline

## 1. Load Data

In [ ]:
dataset = pd.read_csv("loan-train.csv")
print("Shape:", dataset.shape)
dataset.head()

In [ ]:
print("Target distribution:")
print(dataset["Loan_Status"].value_counts())

## 2. Handle Missing Values

In [ ]:
for col in ["Gender", "Married", "Dependents", "Self_Employed"]:
    dataset[col] = dataset[col].fillna(dataset[col].mode()[0])

dataset["LoanAmount"]       = dataset["LoanAmount"].fillna(dataset["LoanAmount"].mean())
dataset["Loan_Amount_Term"] = dataset["Loan_Amount_Term"].fillna(dataset["Loan_Amount_Term"].mode()[0])
dataset["Credit_History"]   = dataset["Credit_History"].fillna(dataset["Credit_History"].mode()[0])

print("Missing values after filling:")
print(dataset.isnull().sum())

## 3. Feature Engineering

Two derived features **must** be computed here AND in `app.py`:
- `LoanAmount_log` = `log(LoanAmount)` — normalises skewed distribution
- `TotalIncome_log` = `log(ApplicantIncome + CoapplicantIncome)`

> **Bug #1 (fixed):** Original app.py sent raw `LoanAmount`; model expects `log(LoanAmount)`.
> **Bug #2 (fixed):** `TotalIncome_log` was never sent by app.py at all.
> **Bug #3 (fixed):** `Property_Area` was at wrong position in the feature array.

In [ ]:
dataset["LoanAmount_log"]  = np.log(dataset["LoanAmount"])
dataset["TotalIncome"]     = dataset["ApplicantIncome"] + dataset["CoapplicantIncome"]
dataset["TotalIncome_log"] = np.log(dataset["TotalIncome"])
dataset[["LoanAmount", "LoanAmount_log", "TotalIncome", "TotalIncome_log"]].describe()

## 4. Encode Categorical Variables

In [ ]:
dataset["Gender"]        = dataset["Gender"].map({"Male":1,"Female":0})
dataset["Married"]       = dataset["Married"].map({"Yes":1,"No":0})
dataset["Education"]     = dataset["Education"].map({"Graduate":1,"Not Graduate":0})
dataset["Self_Employed"] = dataset["Self_Employed"].map({"Yes":1,"No":0})
dataset["Property_Area"] = dataset["Property_Area"].map({"Urban":2,"Semiurban":1,"Rural":0})
dataset["Dependents"]    = dataset["Dependents"].replace("3+", 3).astype(float)
dataset["Loan_Status"]   = dataset["Loan_Status"].map({"Y":1,"N":0})
dataset.head()

## 5. Feature Selection

> The feature list below **must match** the array in `app.py` position for position.

In [ ]:
# 9 features — order must be identical to the array constructed in app.py
FEATURES = [
    "Gender",           # pos 0 — Male=1, Female=0
    "Married",          # pos 1 — Yes=1, No=0
    "Dependents",       # pos 2 — 0/1/2/3
    "Education",        # pos 3 — Graduate=1, Not Graduate=0
    "Loan_Amount_Term", # pos 4 — days (e.g. 360)
    "Credit_History",   # pos 5 — Good=1.0, Bad=0.0
    "LoanAmount_log",   # pos 6 — np.log(LoanAmount)
    "TotalIncome_log",  # pos 7 — np.log(Applicant + CoApplicant income)
    "Property_Area",    # pos 8 — Urban=2, Semiurban=1, Rural=0
]

x = dataset[FEATURES].values
y = dataset["Loan_Status"].values
print("Feature matrix shape:", x.shape)
print("Target shape:", y.shape)

## 6. Train/Test Split and Scaling

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=0
)

sc = StandardScaler()
x_train = sc.fit_transform(x_train)  # fit ONLY on training data
x_test  = sc.transform(x_test)       # use same fitted scaler on test
print("x_train shape:", x_train.shape)
print("x_test shape: ", x_test.shape)

## 7. Train Models

In [ ]:
nb = GaussianNB()
nb.fit(x_train, y_train)
y_pred_nb = nb.predict(x_test)
acc_nb = metrics.accuracy_score(y_test, y_pred_nb)
print(f"Naive Bayes Accuracy: {acc_nb*100:.2f}%")
print()
print(metrics.classification_report(y_test, y_pred_nb))

In [ ]:
dt = DecisionTreeClassifier(criterion="entropy", random_state=0)
dt.fit(x_train, y_train)
y_pred_dt = dt.predict(x_test)
acc_dt = metrics.accuracy_score(y_test, y_pred_dt)
print(f"Decision Tree Accuracy: {acc_dt*100:.2f}%")

In [ ]:
print(f"GaussianNB   : {acc_nb*100:.1f}%")
print(f"Decision Tree: {acc_dt*100:.1f}%")
best_model = nb  # NB is more accurate

## 8. Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, pred) in zip(axes, [("Naive Bayes", y_pred_nb), ("Decision Tree", y_pred_dt)]):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 9. Save model.pkl and scaler.pkl

> **Bug #4 (fixed):** Original notebook saved `DTClassifier` but app.py displayed "Gaussian Naive Bayes" — wrong model was pickled.
> This version correctly saves `GaussianNB`.

In [ ]:
pickle.dump(best_model, open("model.pkl",  "wb"))  # GaussianNB
pickle.dump(sc,         open("scaler.pkl", "wb"))  # StandardScaler (9 features)
print("Saved: model.pkl  -> GaussianNB")
print("Saved: scaler.pkl -> StandardScaler fitted on 9 features")

## 10. Sanity Check — Load and Predict

In [ ]:
loaded_model  = pickle.load(open("model.pkl",  "rb"))
loaded_scaler = pickle.load(open("scaler.pkl", "rb"))

# Perfect applicant: Male, Married, 0 deps, Graduate,
# Term=360, Good credit, LoanAmount=100k, Income=8000+2000=10000, Urban
test_row = np.array([[
    1,                      # Gender
    1,                      # Married
    0,                      # Dependents
    1,                      # Education
    360,                    # Loan_Amount_Term
    1.0,                    # Credit_History
    np.log(100),            # LoanAmount_log
    np.log(10000),          # TotalIncome_log
    2,                      # Property_Area: Urban
]])

scaled_row = loaded_scaler.transform(test_row)
result     = loaded_model.predict(scaled_row)
prob       = loaded_model.predict_proba(scaled_row)[0]

print("Test prediction:", "APPROVED" if result[0] == 1 else "REJECTED")
print(f"Confidence -> Rejected: {prob[0]*100:.1f}% | Approved: {prob[1]*100:.1f}%")